# VisDrone — yolo26**m**

Lam tron mot size: baseline -> prune 50% -> finetune + CWD -> val.
Ket qua la **hai dong** cua bang: `YOLO26-M` va `Ours-M`.

| | |
|---|---|
| Baseline | `yolo26m.pt` (COCO), 100 epoch tren VisDrone |
| Ours | L1-norm uniform 50% (div 8) + 100 epoch CWD tau=9, kd_layers=neck |
| Batch / imgsz / seed | 16 / 640 / 0 |
| cos_lr / patience / warmup | False / 100 / 3.0 |
| Uoc tinh | ~13h, **2 phien** |

> Bon notebook n/s/m/l dung **y het** cac tham so nay. Doi mot cai thoi la ca
> bang het so sanh duoc.

## Cach chay

1. Settings -> Accelerator **GPU T4 x2**, **Internet: On**
2. **Save & Run All**. Lan dau khong can Add Data.
3. Phien tu dung o 10h. Cell cuoi bao con thieu bao nhieu epoch -> Add Data
   output cua chinh lan chay nay roi Save & Run All lai.
4. Xong thi gui lai bang 2 dong o cell cuoi.

Logic nam trong `notebooks/share_visdrone/vd_common.py` trong repo, cell setup
tu `git pull` moi lan chay — sua loi o do la ca nhom co ngay, khong phai import
lai notebook.

Repo: https://github.com/xauskeleton/yolo26_prune_cwd

## 1. Setup

In [1]:
import os, sys, pathlib, subprocess

REPO_DIR = pathlib.Path("/kaggle/working/yolo")
# Luon keo ban moi nhat: logic nam trong repo nen sua o do la ca nhom co ngay.
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/xauskeleton/yolo26_prune_cwd", str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)

# Phai dung fork nay, KHONG "pip install ultralytics": checkpoint sau khi prune
# duoc pickle voi ultralytics.nn.tasks_pruned nen ban chinh thuc khong load duoc.
for d in (REPO_DIR, REPO_DIR / "pruning", REPO_DIR / "notebooks" / "share_visdrone"):
    sys.path.insert(0, str(d))
# DDP sinh tien trinh con chay file tam ngoai repo -> phai truyen qua PYTHONPATH.
os.environ["PYTHONPATH"] = str(REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", "requirements.txt"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".",
                "--no-deps"], check=False)

import torch
import vd_common as V
print("GPU:", torch.cuda.device_count(), "| vd_common:", V.__file__)

Cloning into '/kaggle/working/yolo'...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.1/150.1 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.6.0 which is incompatible.


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
GPU: 2 | vd_common: /kaggle/working/yolo/notebooks/share_visdrone/vd_common.py


## 2. Cau hinh

In [2]:
SIZE   = "m"
MODEL  = "yolo26m.pt"
RATIO  = 0.5

BASE_NAME = "vd_yolo26m"
OURS_NAME = "vd_oursm"
PRUNED = REPO_DIR / "weights" / "yolo26m_vd_pruned50.pt"

V.init(
    REPO_DIR = REPO_DIR,
    DATA     = "VisDrone.yaml",   # Ultralytics tu tai 2.3 GB, can Internet: On
    EPOCHS   = 100,
    BATCH    = 16,
    IMGSZ    = 640,
    DEVICE   = "0,1" if torch.cuda.device_count() > 1 else "0",
    # Chot cung thay vi de mac dinh: cac run VOC truoc day khong dong nhat
    # (n/s/l dung batch 32 + cos_lr=True + patience 20-30, m dung 16/False/100).
    COS_LR   = False,
    PATIENCE = 100,
    WARMUP   = 3.0,
    # Kaggle giet phien o 12h va phien bi giet thi KHONG luu output.
    STOP_AFTER_H = 10.0,
)

# Chi dung khi chi co moi mot file last.pt roi le (phien bi danh dau failed).
# Upload ca thu muc <run_name>/weights/ thi khong can dien gi.
MANUAL_LAST = {BASE_NAME: "", OURS_NAME: ""}

print(BASE_NAME, "|", OURS_NAME, "|", V.CFG["DEVICE"])

vd_yolo26m | vd_oursm | 0,1


## 3. Resume

In [3]:
V.restore(BASE_NAME, OURS_NAME, PRUNED, MANUAL_LAST)

  vd_yolo26m: 100 epoch  <- /kaggle/input/datasets/chimbellll/resume/vd_resume_m/vd_yolo26m
  vd_oursm: 80 epoch  <- /kaggle/input/datasets/chimbellll/resume/vd_resume_m/vd_oursm

baseline : 100 / 100 epoch
ours     : 80 / 100 epoch


## 4. Baseline yolo26m

In [4]:
n_base = V.train(BASE_NAME, MODEL)

if n_base < 100:
    print()
    print("Baseline moi {}/100 epoch - het gio phien nay.".format(n_base))
    print("Add Data output lan nay roi Save & Run All lai. Cac cell duoi bo qua.")

[vd_yolo26m] da du 100/100 epoch, bo qua


## 5. Prune 50%

In [5]:
BEST_BASE = REPO_DIR / "runs" / BASE_NAME / "weights" / "best.pt"

if n_base < 100:
    print("Bo qua: baseline chua xong.")
else:
    V.prune50(SIZE, BEST_BASE, PRUNED, RATIO)

Step 1: Thu thập BatchNorm layers...
  Tổng BN layers: 124
  Ignore (residual): 34
  Chunk constraint: 0
  Prunable BN layers: 90

Step 6: Tạo pruned model config...
  nc: 10, scale: m
  Backbone layers: 11
  Head layers: 13
    [ 0] n=1 Conv                 args=[64, 3, 2]
    [ 1] n=1 Conv                 args=[128, 3, 2]
    [ 2] n=1 C3k2Pruned           args=[256, True]
    [ 3] n=1 Conv                 args=[256, 3, 2]
    [ 4] n=1 C3k2Pruned           args=[512, True]
    [ 5] n=1 Conv                 args=[512, 3, 2]
    [ 6] n=1 C3k2Pruned           args=[512, True]
    [ 7] n=1 Conv                 args=[1024, 3, 2]
    [ 8] n=1 C3k2Pruned           args=[1024, True]
    [ 9] n=1 SPPFPruned           args=[1024, 5, 3, True]
    [10] n=1 C2PSAPruned          args=[1024]
    [11] n=1 nn.Upsample          args=['None', 2, 'nearest']
    [12] n=1 Concat               args=[1]
    [13] n=1 C3k2Pruned           args=[512, True]
    [14] n=1 nn.Upsample          args=['None', 2, 'nea

## 6. Finetune + CWD

In [6]:
if n_base < 100 or not PRUNED.exists():
    print("Bo qua: chua co model da prune.")
    n_ours = 0
else:
    # Teacher la baseline cua chinh size nay.
    n_ours = V.train(OURS_NAME, str(PRUNED),
                     finetune=True,      # build DetectionModelPruned tu maskbndict
                     kd=True, kd_teacher=str(BEST_BASE), kd_method="cwd",
                     kd_lambda=0.5, kd_layers="neck", kd_warmup=5,
                     cwd_temperature=9.0)

[vd_oursm] resume tu epoch 80
New https://pypi.org/project/ultralytics/8.4.163 available 😃 Update with 'pip install -U ultralytics'
[Resume] Found custom_training_args in checkpoint: ['sr', 'dms', 'dms_target', 'dms_lambda', 'dms_lr', 'dms_taylor_type', 'dms_decay_ratio', 'dms_grad_scale', 'finetune', 'kd', 'kd_teacher', 'kd_lambda', 'cwd_temperature', 'kd_layers', 'kd_warmup', 'cwd_learnable_tau_lr', 'cwd_learnable_tau_init', 'kd_method', 'mgd_mask_ratio', 'fitnets_normalize']
Ultralytics 8.4.14 🚀 Python-3.12.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 14912MiB)
                                                      CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, cwd_learnable_tau_init=9.0, cwd_learnable_tau_lr=0.001, cwd_projection=Fal

## 7. Ket qua

In [7]:
V.report(SIZE, BASE_NAME, OURS_NAME)

Ultralytics 8.4.14 🚀 Python-3.12.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 14912MiB)
YOLO26m summary (fused): 132 layers, 20,357,162 parameters, 0 gradients, 67.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2585.6±689.6 MB/s, size: 131.3 KB)
val: Scanning /kaggle/working/datasets/VisDrone/labels/val.cache... 548 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 548/548 191.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 2.4it/s 14.8s
                   all        548      38759      0.573      0.445      0.469      0.287
            pedestrian        520       8844      0.644      0.504      0.556      0.273
                people        482       5125      0.579      0.397      0.431      0.183
               bicycle        364       1287      0.399      0.222      0.211     0.0957
                   car        515      14064      0.771      0.817      0.842      0.612
                   van     